# V5W_07 - Geometria di Riemann: asse fenotipico + decoding baseline

Le matrici di connettivita/covarianza sono SPD -> varieta di Riemann. Si usa la geometria corretta (covarianze + tangent space) per:
1. **Asse fenotipico**: media di Riemann per soggetto -> tangent space -> PCA. Se i soggetti si **allineano** su PC1, il continuum e una geodetica. Correlazione Tangent-PC1 vs PI euclideo vs alpha mu.
2. **Decoding baseline**: MDM + TangentSpace+LR (gold-standard EEG classico), subject-specific LOSO. Se chance -> ceiling blindato.

Covarianza per trial: estimatore OAS (SPD ben condizionata). **Env: `daniele_311`** (pyriemann sul server). Richiede cache V5W_05 (PI) e V5W_06 (alpha, opzionale).

## par.1 - Config + PI/alpha dalle cache

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_riemann
from pyriemann.classification import MDM

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w07')
project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
V5W05    = project_root / 'models' / 'v5w05'
V5W06    = project_root / 'models' / 'v5w06'
CKPT_DIR = project_root / 'models' / 'v5w07'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_ROOT = project_root / 'data' / '5words_subjects'
N_CHAN, FS, N_CLASSES = 61, 256, 5
triu_idx = np.triu_indices(N_CHAN, k=1)
word2label = json.loads((project_root/'configs'/'label_schemes'/'label2idx_5words.json').read_text())
_PAT = re.compile(r'^P(\d+)_S(\d+)$')

_feat = np.load(V5W05/'feat_abs_pcc.npz', allow_pickle=True)
FEAT_G, SUBJ_F = _feat['feat_g'], _feat['subj'].tolist()
_lab = np.load(V5W05/'cluster_labels.npz', allow_pickle=True)
lab_map = {int(s): int(l) for s, l in zip(_lab['subj'], _lab['labels'])}
def vec_to_sym(v):
    M = np.zeros((N_CHAN, N_CHAN)); M[triu_idx] = v; return M + M.T
NS = np.array([vec_to_sym(FEAT_G[i]).sum(1)/(N_CHAN-1) for i in range(len(SUBJ_F))])
lab_arr = np.array([lab_map[s] for s in SUBJ_F])
_diff = NS[lab_arr==1].mean(0) - NS[lab_arr==0].mean(0); _diff /= (np.linalg.norm(_diff)+1e-12)
PI = {s: float(p) for s, p in zip(SUBJ_F, NS @ _diff)}
ALPHA = {}
if (V5W06/'subject_alpha.npz').exists():
    z = np.load(V5W06/'subject_alpha.npz', allow_pickle=True)
    ALPHA = {int(s): float(a) for s, a in zip(z['subj'], z['alpha'])}
log.info(f'PI: {len(PI)} sogg.  alpha: {len(ALPHA)} sogg.')


## par.2 - Covarianze per-trial (SPD)

In [ ]:
CACHE = CKPT_DIR / 'covs.npz'
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    COVS, SUBJ, SESS, LAB = z['covs'], z['subj'], z['sess'], z['lab']
    log.info(f'Covarianze da cache: {COVS.shape}')
else:
    cov_est = Covariances(estimator='oas')
    covs_all, subj_all, sess_all, lab_all = [], [], [], []
    sdirs = sorted(d for d in CSV_ROOT.iterdir() if _PAT.match(d.name))
    for sd in tqdm(sdirs, desc='covarianze'):
        m = _PAT.match(sd.name); sid, ses = int(m.group(1)), int(m.group(2))
        X, y = [], []
        for csv in sorted(sd.glob('*_img_*.csv')):
            if csv.name.startswith('._'): continue
            w = csv.name.split('_img_')[0]
            if w not in word2label: continue
            arr = pd.read_csv(csv, header=None).values.astype(np.float32)
            if arr.shape != (N_CHAN, 384): continue
            X.append(arr); y.append(word2label[w])
        if not X: continue
        C = cov_est.transform(np.stack(X))
        covs_all.append(C.astype(np.float32))
        subj_all += [sid]*len(y); sess_all += [ses]*len(y); lab_all += y
    COVS = np.concatenate(covs_all); SUBJ = np.array(subj_all); SESS = np.array(sess_all); LAB = np.array(lab_all)
    np.savez(CACHE, covs=COVS, subj=SUBJ, sess=SESS, lab=LAB)
    log.info(f'Covarianze calcolate e salvate: {COVS.shape}')
ALL_SUBJ = sorted(set(SUBJ.tolist()))
print(f'Trial: {len(COVS)}  soggetti: {len(ALL_SUBJ)}')


## par.3 - Asse fenotipico Riemann (allineamento)

In [ ]:
subj_mean = []
for sid in tqdm(ALL_SUBJ, desc='media Riemann/sogg'):
    subj_mean.append(mean_riemann(COVS[SUBJ == sid]))
M = np.stack(subj_mean)
ts = TangentSpace().fit(M)
TS = ts.transform(M)
Z = PCA(n_components=5, random_state=42).fit_transform(StandardScaler().fit_transform(TS))

pi_arr = np.array([PI.get(s, np.nan) for s in ALL_SUBJ])
if np.corrcoef(Z[:, 0], np.nan_to_num(pi_arr))[0, 1] < 0:
    Z[:, 0] = -Z[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sc = axes[0].scatter(Z[:, 0], Z[:, 1], c=pi_arr, cmap='RdBu_r', s=90, edgecolor='k', lw=0.5)
for s, zx, zy in zip(ALL_SUBJ, Z[:, 0], Z[:, 1]):
    axes[0].annotate(f'P{s}', (zx, zy), fontsize=6, alpha=0.6)
axes[0].set_xlabel('Tangent PC1'); axes[0].set_ylabel('Tangent PC2')
axes[0].set_title('Soggetti nel tangent space (Riemann), colore = PI', fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='PI')

m = ~np.isnan(pi_arr)
r_pi, p_pi = pearsonr(Z[m, 0], pi_arr[m])
axes[1].scatter(pi_arr[m], Z[m, 0], s=70, edgecolor='k', lw=0.5, color='#444')
axes[1].set_xlabel('PI euclideo (triu)'); axes[1].set_ylabel('Tangent PC1 (Riemann)')
axes[1].set_title(f'Riemann vs Euclideo: r={r_pi:.3f} (p={p_pi:.1e})', fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_riemann_axis.png', dpi=160, bbox_inches='tight'); plt.show()

print('='*60)
print(f'  Tangent-PC1 vs PI euclideo:  r={r_pi:+.3f}  p={p_pi:.2e}')
if ALPHA:
    al = np.array([ALPHA.get(s, np.nan) for s in ALL_SUBJ]); ma = ~np.isnan(al)
    r_a, p_a = pearsonr(Z[ma, 0], al[ma])
    print(f'  Tangent-PC1 vs alpha mu:     r={r_a:+.3f}  p={p_a:.2e}')
print('  -> PC1 allineato con PI e alpha => asse confermato in 3 modi indipendenti')
print('='*60)
np.savez(CKPT_DIR/'tangent_axis.npz', subj=np.array(ALL_SUBJ), pc1=Z[:, 0], pc=Z)


## par.4 - Decoding baseline Riemann

In [ ]:
rows = []
for sid in tqdm(ALL_SUBJ, desc='Riemann decoding'):
    msk = SUBJ == sid
    covs_s, lab_s, sess_s = COVS[msk], LAB[msk], SESS[msk]
    us = sorted(set(sess_s.tolist()))
    if len(us) < 2:
        continue
    te = us[-1]
    tr_m, te_m = sess_s != te, sess_s == te
    if tr_m.sum() == 0 or te_m.sum() == 0:
        continue
    Xtr, ytr, Xte, yte = covs_s[tr_m], lab_s[tr_m], covs_s[te_m], lab_s[te_m]
    try:
        b_mdm = balanced_accuracy_score(yte, MDM().fit(Xtr, ytr).predict(Xte))
    except Exception:
        b_mdm = np.nan
    try:
        tslr = make_pipeline(TangentSpace(), LogisticRegression(max_iter=1000, C=1.0))
        tslr.fit(Xtr, ytr)
        b_ts = balanced_accuracy_score(yte, tslr.predict(Xte))
    except Exception:
        b_ts = np.nan
    rows.append((sid, b_mdm, b_ts))

df = pd.DataFrame(rows, columns=['subj', 'MDM', 'TS_LR'])
chance = 1 / N_CLASSES
import torch
dh = project_root/'models'/'v5w03'
b_dh = np.array([float(torch.load(c, weights_only=False)['test_bacc']) for c in sorted(dh.glob('P*.pt'))]) if dh.exists() else np.array([])

print('='*60)
print(f'  V5W_07 - Decoding Riemann subject-specific ({len(df)} sogg., chance {chance:.0%})')
print('='*60)
for col in ['MDM', 'TS_LR']:
    v = df[col].dropna().values
    print(f'  {col:6s}: mean={v.mean():.4f}  median={np.median(v):.4f}  max={v.max():.4f}  >chance={(v>chance).mean()*100:.0f}%')
if len(b_dh):
    print(f'  DHSLP : mean={b_dh.mean():.4f}  (V5W_03, riferimento)')
print('  -> se anche MDM/TS-LR (gold-standard EEG classico) sono chance, ceiling blindato')
print('='*60)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(np.arange(len(df))-0.2, df['MDM'], 0.4, label='MDM', color='#1f77b4', alpha=0.8)
ax.bar(np.arange(len(df))+0.2, df['TS_LR'], 0.4, label='TS+LR', color='#ff7f0e', alpha=0.8)
ax.axhline(chance, color='k', ls='--', lw=1.5, label=f'Chance ({chance:.0%})')
ax.set_xticks(range(len(df))); ax.set_xticklabels([f'P{s}' for s in df['subj']], rotation=90, fontsize=6)
ax.set_ylabel('Balanced Accuracy'); ax.set_title('V5W_07 - Riemann decoding (5 parole)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_riemann_decoding.png', dpi=150, bbox_inches='tight'); plt.show()
df.to_csv(FIG_DIR/'v5w07_riemann_decoding.csv', index=False)


## par.5 - La separazione e reale? Permutation test nello spazio tangente

Lo scatter (par.3) mostra due blob con un vuoto. Test rigoroso: silhouette k=2
sulle feature tangenti (TS) vs distribuzione nulla (feature permutate per-colonna),
+ sweep k=2..6. Se k=2 e significativo, la geometria di Riemann rivela una
dicotomia che l'analisi euclidea (V5W_05, p=0.62) aveva mascherato.

In [ ]:
# par.5 - Permutation test sulla silhouette nello spazio tangente (TS da par.3)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
def _sil_k(feat, k):
    Z2 = PCA(n_components=min(20, feat.shape[0]-1), random_state=42).fit_transform(StandardScaler().fit_transform(feat))
    lab = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Z2)
    return silhouette_score(Z2, lab), lab
def _perm(feat, k, nperm=1000, seed=42):
    obs, _ = _sil_k(feat, k); rng = np.random.default_rng(seed); n, p = feat.shape; null = np.empty(nperm)
    for i in range(nperm):
        idx = rng.random((n, p)).argsort(axis=0)
        null[i], _ = _sil_k(np.take_along_axis(feat, idx, axis=0), k)
    return obs, null, (null >= obs).mean()

print('Permutation test su spazio tangente (Riemann):')
obs2, null2, p2 = _perm(TS, 2)
print(f'  k=2: silhouette={obs2:.3f}  null={null2.mean():.3f}+/-{null2.std():.3f}  p={p2:.3f}  {"*SIGNIFICATIVO*" if p2<0.05 else "n.s."}')
print('  sweep k:')
best=(None,-9)
for k in range(2,7):
    o,nu,pp=_perm(TS,k,nperm=300)
    gap=o-nu.mean(); flag=' *' if pp<0.05 else ''
    print(f'    k={k}: sil={o:.3f} null={nu.mean():.3f} gap={gap:+.3f} p={pp:.3f}{flag}')
    if gap>best[1]: best=(k,gap)

fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(null2, bins=30, color='0.7', edgecolor='white')
ax.axvline(obs2, color='#d62728', lw=2.5, label=f'osservato={obs2:.3f}')
ax.axvline(np.percentile(null2,95), color='k', ls='--', lw=1.2, label='null p95')
ax.set_title(f'Spazio tangente Riemann - k=2  p={p2:.3f}', fontweight='bold')
ax.set_xlabel('silhouette k=2'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_tangent_permutation.png', dpi=160, bbox_inches='tight'); plt.show()
print('='*60)
if p2<0.05:
    print('  => DICOTOMIA REALE nello spazio di Riemann (euclideo la mascherava).')
    print('     I 2 fenotipi esistono, serviva la geometria corretta.')
else:
    print('  => k=2 non significativo nemmeno in Riemann: il continuum regge,')
    print('     il vuoto nello scatter era rumore visivo.')
print('='*60)


## par.6 - Caratterizzazione dei due cluster (Riemann)

Estrae le label k=2 reali nello spazio tangente, le allinea con PI/alpha, esegue
i check anti-artefatto (norma/traccia covarianza, distanza geodetica) e mappa
*cosa* distingue i fenotipi: connettivita (node strength |PCC|) + spettro per banda.

In [ ]:
# === par.6a - Label k=2 reali: alignment, anti-artefatto, geodetica ===
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu
from pyriemann.utils.distance import distance_riemann

Zt = PCA(n_components=min(20, TS.shape[0]-1), random_state=42).fit_transform(StandardScaler().fit_transform(TS))
LABt = KMeans(n_clusters=2, n_init=30, random_state=42).fit_predict(Zt)
SUBJ_ARR = np.array(ALL_SUBJ)
pi_a = np.array([PI.get(s, np.nan) for s in ALL_SUBJ])
if np.nanmean(pi_a[LABt == 1]) < np.nanmean(pi_a[LABt == 0]): LABt = 1 - LABt
print(f'Cluster C0: n={(LABt==0).sum()}   C1: n={(LABt==1).sum()}')
print(f'  C0 = {sorted(SUBJ_ARR[LABt==0].tolist())}')
print(f'  C1 = {sorted(SUBJ_ARR[LABt==1].tolist())}')

def cmp(name, arr, fmt='.3f'):
    a0, a1 = arr[LABt==0], arr[LABt==1]; a0=a0[~np.isnan(a0)]; a1=a1[~np.isnan(a1)]
    u, p = mannwhitneyu(a0, a1)
    print(f'  {name:26s}: C0={a0.mean():{fmt}}  C1={a1.mean():{fmt}}  MW p={p:.4f}')

print('\nAllineamento (DEVONO differire = la dicotomia e sostanziale):')
cmp('PI euclideo', pi_a)
if ALPHA: cmp('alpha mu', np.array([ALPHA.get(s, np.nan) for s in ALL_SUBJ]))

print('\nAnti-artefatto (NON dovrebbero guidare lo split = no qualita/ampiezza):')
covnorm = np.array([np.linalg.norm(M[i], 'fro') for i in range(len(ALL_SUBJ))])
covtr   = np.array([np.trace(M[i]) for i in range(len(ALL_SUBJ))])
cmp('Frobenius norm cov', covnorm, '.1f')
cmp('trace cov (pot. tot)', covtr, '.1f')
print('  (la metrica affine-invariante ignora la scala globale: anche se norma/traccia')
print('   differiscono, lo split e per STRUTTURA, non per ampiezza.)')

m0 = mean_riemann(M[LABt==0]); m1 = mean_riemann(M[LABt==1])
d_bw = distance_riemann(m0, m1)
dw0 = np.mean([distance_riemann(M[i], m0) for i in np.where(LABt==0)[0]])
dw1 = np.mean([distance_riemann(M[i], m1) for i in np.where(LABt==1)[0]])
print(f'\nDistanza geodetica tra cluster: {d_bw:.3f}  (within C0={dw0:.3f} C1={dw1:.3f})')
print(f'Separazione between/within: {d_bw/((dw0+dw1)/2):.2f}')
np.savez(CKPT_DIR/'tangent_clusters.npz', subj=SUBJ_ARR, label=LABt)


In [ ]:
# === par.6b - Cosa distingue i 2 cluster: connettivita + spettro per banda ===
import mne
from scipy.signal import welch
ELOC = project_root/'src'/'io'/'ebneuro.locs'
_RN={'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}; _BD={'A1','A2'}
_mt=mne.channels.read_custom_montage(str(ELOC), coord_frame='head'); _ps=_mt.get_positions()['ch_pos']
CHAN_NAMES=[_RN.get(c,c) for c in _mt.ch_names if c not in _BD][:N_CHAN]
CH_POS={_RN.get(c,c):_ps[c] for c in _mt.ch_names if c not in _BD}
def topo_info():
    info=mne.create_info(CHAN_NAMES,256,'eeg')
    info.set_montage(mne.channels.make_dig_montage(ch_pos={c:CH_POS[c] for c in CHAN_NAMES},coord_frame='head'),on_missing='warn')
    return info

# connettivita: node strength |PCC| per soggetto (da FEAT_G), label tangenti
NS_map={SUBJ_F[i]: vec_to_sym(FEAT_G[i]).sum(1)/(N_CHAN-1) for i in range(len(SUBJ_F))}
idx_ns=[i for i,s in enumerate(ALL_SUBJ) if s in NS_map]
ns=np.array([NS_map[ALL_SUBJ[i]] for i in idx_ns]); labns=LABt[idx_ns]
ns_diff=ns[labns==1].mean(0)-ns[labns==0].mean(0)

# spettro: relative band power per soggetto (cache)
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,13),'beta':(13,30),'gamma':(30,45)}
SPC=CKPT_DIR/'subject_bandpow.npz'
if SPC.exists():
    z=np.load(SPC,allow_pickle=True); BP={int(s):p for s,p in zip(z['subj'],z['bp'])}
else:
    sc=defaultdict(list)
    for sd in sorted(CSV_ROOT.iterdir()):
        mm=_PAT.match(sd.name)
        if mm:
            for c in sd.glob('*_img_*.csv'):
                if not c.name.startswith('._'): sc[int(mm.group(1))].append(c)
    BP={}
    for sid in tqdm(sorted(sc), desc='band power'):
        ps=None;n=0
        for p in sc[sid]:
            x=pd.read_csv(p,header=None).values.astype(np.float32)
            f,Pxx=welch(x,fs=256,nperseg=256,axis=1); ps=Pxx if ps is None else ps+Pxx; n+=1
        ps/=n; tot=ps[:,(f>=1)&(f<45)].sum(1)+1e-12
        BP[sid]=np.stack([ps[:,(f>=lo)&(f<hi)].sum(1)/tot for lo,hi in BANDS.values()])
    np.savez(SPC,subj=np.array(list(BP)),bp=np.array(list(BP.values())))
idx_bp=[i for i,s in enumerate(ALL_SUBJ) if s in BP]
BPa=np.array([BP[ALL_SUBJ[i]] for i in idx_bp]); labbp=LABt[idx_bp]

info=topo_info()
fig,axes=plt.subplots(1,6,figsize=(20,3.7))
dm=np.abs(ns_diff).max()+1e-12
mne.viz.plot_topomap(ns_diff,info,axes=axes[0],show=False,cmap='RdBu_r',vlim=(-dm,dm),contours=4)
axes[0].set_title('|PCC| node str.\nC1-C0',fontweight='bold',fontsize=10)
for k,bn in enumerate(BANDS):
    d=BPa[labbp==1,k,:].mean(0)-BPa[labbp==0,k,:].mean(0); dmx=np.abs(d).max()+1e-12
    mne.viz.plot_topomap(d,info,axes=axes[k+1],show=False,cmap='RdBu_r',vlim=(-dmx,dmx),contours=4)
    axes[k+1].set_title(f'{bn}\nC1-C0',fontweight='bold',fontsize=10)
fig.suptitle('V5W_07 par.6b - Cosa distingue i 2 cluster Riemann (rosso = C1>C0)',fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_cluster_diff.png',dpi=160,bbox_inches='tight'); plt.show()

def cohend(a,b):
    na,nb=len(a),len(b); sp=np.sqrt(((na-1)*a.var(0,ddof=1)+(nb-1)*b.var(0,ddof=1))/(na+nb-2)); return (b.mean(0)-a.mean(0))/(sp+1e-12)
print('Cohen d (C1-C0) per banda (elettrodo di picco):')
for k,bn in enumerate(BANDS):
    d=cohend(BPa[labbp==0,k,:],BPa[labbp==1,k,:]); j=np.argmax(np.abs(d))
    print(f'  {bn:6s}: d_max={d[j]:+.2f} @ {CHAN_NAMES[j]}')


## par.7 - Edge discriminanti + decodabilita per fenotipo

(a) Quali CONNESSIONI (coppie di elettrodi) distinguono i due cluster (Cohen d sugli archi |PCC|).
(b) I due fenotipi decodificano diversamente? Ponte tra ceiling (Cap.5) e fenotipi (Cap.6) -> ipotesi BCI-literate.

In [ ]:
# === par.7a - Edge discriminanti tra i due cluster (|PCC|) ===
lab_by_subj = {int(SUBJ_ARR[i]): int(LABt[i]) for i in range(len(SUBJ_ARR))}
g_lab = np.array([lab_by_subj.get(int(s), -1) for s in SUBJ_F])
G = FEAT_G[g_lab >= 0]; gl = g_lab[g_lab >= 0]
a, b = G[gl == 0], G[gl == 1]
sp = np.sqrt(((len(a)-1)*a.var(0,ddof=1) + (len(b)-1)*b.var(0,ddof=1)) / (len(a)+len(b)-2))
d_edge = (b.mean(0) - a.mean(0)) / (sp + 1e-12)          # (1830,) Cohen d per arco, C1-C0

import mne
ELOC = project_root/'src'/'io'/'ebneuro.locs'
_RN={'T3':'T7','T4':'T8','T5':'P7','T6':'P8'}; _BD={'A1','A2'}
_mt=mne.channels.read_custom_montage(str(ELOC), coord_frame='head'); _ps=_mt.get_positions()['ch_pos']
CHAN_NAMES=[_RN.get(c,c) for c in _mt.ch_names if c not in _BD][:N_CHAN]
P2D=np.array([_ps[[k for k,v in _RN.items() if v==c] [0] if c in _RN.values() and c not in _ps else c][:2]
              if False else _ps.get(c, _ps[[kk for kk in _ps][0]])[:2] for c in CHAN_NAMES])
# fallback robusto per posizioni 2D
P2D=np.array([(_ps[c][:2] if c in _ps else (0.0,0.0)) for c in CHAN_NAMES])

order=np.argsort(-np.abs(d_edge)); TOPK=25
fig, ax=plt.subplots(figsize=(7,7))
th=np.linspace(0,2*np.pi,200); R=np.abs(P2D).max()*1.1
ax.plot(R*np.cos(th), R*np.sin(th), color='0.8', lw=1)
ax.scatter(P2D[:,0], P2D[:,1], s=60, c='0.5', zorder=3)
for k in order[:TOPK]:
    i,j = triu_idx[0][k], triu_idx[1][k]
    col = '#d62728' if d_edge[k] > 0 else '#1f77b4'
    ax.plot([P2D[i,0],P2D[j,0]],[P2D[i,1],P2D[j,1]], color=col, lw=1.5*abs(d_edge[k]), alpha=0.7, zorder=2)
ax.set_title(f'Top-{TOPK} edge discriminanti (rosso=C1>C0, blu=C0>C1)', fontweight='bold')
ax.set_aspect('equal'); ax.axis('off')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_discriminant_edges.png', dpi=160, bbox_inches='tight'); plt.show()

print('Top-12 edge per |Cohen d| (C1-C0):')
for k in order[:12]:
    i,j = triu_idx[0][k], triu_idx[1][k]
    print(f'  {CHAN_NAMES[i]:4s} - {CHAN_NAMES[j]:4s}  d={d_edge[k]:+.2f}')


In [ ]:
# === par.7b - Decodabilita per fenotipo (ponte ceiling x fenotipi) ===
import torch
from scipy.stats import mannwhitneyu
lab_by_subj = {int(SUBJ_ARR[i]): int(LABt[i]) for i in range(len(SUBJ_ARR))}
chance = 1 / N_CLASSES

dh = {int(c.stem[1:]): float(torch.load(c, weights_only=False)['test_bacc'])
      for c in sorted((project_root/'models'/'v5w03').glob('P*.pt'))}
methods = {'DHSLP': dh}
csvp = FIG_DIR/'v5w07_riemann_decoding.csv'
if csvp.exists():
    rd = pd.read_csv(csvp)
    methods['MDM']   = {int(r.subj): float(r.MDM)   for r in rd.itertuples() if not np.isnan(r.MDM)}
    methods['TS+LR'] = {int(r.subj): float(r.TS_LR) for r in rd.itertuples() if not np.isnan(r.TS_LR)}

rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, len(methods), figsize=(5*len(methods), 5), squeeze=False)
print('Decodabilita per cluster (Mann-Whitney C0 vs C1):')
for ax, (mname, dd) in zip(axes[0], methods.items()):
    b0 = np.array([v for s, v in dd.items() if lab_by_subj.get(s) == 0])
    b1 = np.array([v for s, v in dd.items() if lab_by_subj.get(s) == 1])
    u, p = mannwhitneyu(b0, b1) if len(b0) and len(b1) else (np.nan, np.nan)
    for i, (v, col) in enumerate([(b0, '#4DA3FF'), (b1, '#FF8C42')]):
        jit = i + (rng.random(len(v)) - 0.5) * 0.2
        ax.scatter(jit, v, color=col, s=40, edgecolor='w', linewidth=0.5, zorder=3)
        ax.boxplot(v, positions=[i], widths=0.5, showfliers=False, medianprops=dict(color=col, lw=2))
    ax.axhline(chance, color='k', ls='--', lw=1.2, label=f'chance {chance:.0%}')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['C0', 'C1']); ax.set_ylabel('bAcc')
    ax.set_title(f'{mname}\nC0={b0.mean():.3f} C1={b1.mean():.3f} p={p:.3f}', fontweight='bold'); ax.legend(fontsize=8)
    print(f'  {mname:6s}: C0={b0.mean():.4f} (n={len(b0)})  C1={b1.mean():.4f} (n={len(b1)})  p={p:.4f}')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_decodability_by_cluster.png', dpi=160, bbox_inches='tight'); plt.show()
print('-'*60)
print('  significativo -> il fenotipo predice la performance BCI (literate)')
print('  nessuna differenza, entrambi ~chance -> fenotipi = tipi di connettivita/spettro,')
print('  indipendenti dalla decodabilita (floor per tutti)')


## par.8 - DECISIVO: la dicotomia e struttura o ampiezza?

La par.6a ha rivelato che i due cluster differiscono 2.6x in ampiezza (Frobenius
norm p<1e-4). La distanza geodetica affine-invariante NON e invariante allo
scaling per-matrice: gran parte della separazione potrebbe essere ampiezza.
Test: normalizzo ogni covarianza-soggetto a **det=1** (rimozione di scala corretta
per la metrica affine-invariante) e ri-testo k=2. Se sopravvive -> strutturale;
se crolla -> era ampiezza (confondente di qualita/gain, punto rosso #3).

In [ ]:
# === par.8 - Dicotomia dopo normalizzazione di scala (det=1) ===
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

def _sil_k(feat, k):
    Z2 = PCA(n_components=min(20, feat.shape[0]-1), random_state=42).fit_transform(StandardScaler().fit_transform(feat))
    lab = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(Z2)
    return silhouette_score(Z2, lab), lab
def _perm(feat, k, nperm=1000, seed=42):
    obs, _ = _sil_k(feat, k); rng = np.random.default_rng(seed); n, p = feat.shape; null = np.empty(nperm)
    for i in range(nperm):
        idx = rng.random((n, p)).argsort(axis=0)
        null[i], _ = _sil_k(np.take_along_axis(feat, idx, axis=0), k)
    return obs, null, (null >= obs).mean()

# label originali (con ampiezza): ricalcola da TS se non in memoria
if 'LABt' not in dir():
    _, LABt = _sil_k(TS, 2)
    log.info('LABt ricalcolato da TS (par.6a non eseguita in questa sessione)')

# normalizza ogni media-soggetto a det=1 (rimuove la scala globale)
def det1(C):
    s, ld = np.linalg.slogdet(C)
    return C * np.exp(-ld / C.shape[0])
Mn = np.stack([det1(M[i]) for i in range(len(M))])
print(f'Check det dopo norm: {np.linalg.slogdet(Mn[0])[1]:.2e} (atteso ~0)')

tsn = TangentSpace().fit(Mn)
TSn = tsn.transform(Mn)
obs_n, null_n, p_n = _perm(TSn, 2)
_, LABn = _sil_k(TSn, 2)
ari = adjusted_rand_score(LABn, LABt)

print('='*60)
print('  PRIMA (con ampiezza):  silhouette k=2 = 0.376   p<0.001')
print(f'  DOPO  (det=1, scala rimossa): silhouette k=2 = {obs_n:.3f}  null={null_n.mean():.3f}  p={p_n:.3f}')
print(f'  ARI(label dopo, label prima) = {ari:.3f}')
print('-'*60)
if p_n < 0.05 and ari > 0.5:
    print('  => SOPRAVVIVE: la dicotomia e STRUTTURALE (shape), non ampiezza. Blindato.')
elif p_n < 0.05:
    print('  => significativa ma label diverse (ARI basso): struttura c e ma NON la stessa')
    print('     bipartizione -> rivedere; lo split originale era in parte ampiezza.')
else:
    print('  => CROLLA: la dicotomia era guidata dall AMPIEZZA (gain/qualita).')
    print('     Confondente confermato (punto rosso #3). Da riportare onestamente.')
print('='*60)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(null_n, bins=30, color='0.7', edgecolor='white')
ax.axvline(obs_n, color='#d62728', lw=2.5, label=f'osservato={obs_n:.3f}')
ax.axvline(np.percentile(null_n, 95), color='k', ls='--', lw=1.2, label='null p95')
ax.set_title(f'Dopo det=1 (scala rimossa) - k=2  p={p_n:.3f}', fontweight='bold')
ax.set_xlabel('silhouette k=2'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_detnorm_permutation.png', dpi=160, bbox_inches='tight'); plt.show()
